# 04 - Battery Dispatch & Cost Optimization
## Linear Program Formulation, Rolling Simulation, and Strategy Comparison

This notebook scales historical time series to a commercial facility site (5 MW peak load, 5 MW solar PV) and simulates four battery dispatch strategies:
1. No Battery Baseline
2. Rule-Based Strategy (heuristic peak discharge / off-peak charge)
3. Forecast-Informed Strategy (rolling 24h LP via HiGHS using LightGBM generation forecast)
4. Perfect Foresight Reference (upper bound LP with actual generation)


In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src.preprocessing import get_clean_data
from src.battery_simulator import BatteryConfig
from src.strategies import scale_to_site, compare_dispatch_strategies

clean_df = get_clean_data()
preds_df = pd.read_parquet('../reports/test_predictions.parquet')

# Align chronological test slice
test_slice = clean_df.loc[preds_df.index]
site_df = scale_to_site(test_slice, peak_load_mw=5.0, solar_capacity_mw=5.0)

# Scaled solar forecast
solar_fc_scaled = preds_df['model_lightgbm'] * site_df['solar_scale_factor']

battery_cfg = BatteryConfig(
    energy_capacity_mwh=10.0,
    max_charge_power_mw=2.5,
    max_discharge_power_mw=2.5,
    round_trip_efficiency=0.88,
    soc_min=0.10,
    soc_max=0.90,
    grid_import_limit_mw=6.0,
    grid_export_limit_mw=5.0,
    retail_adder_eur_mwh=20.0,
    export_ratio=1.0,
    degradation_cost_eur_mwh=0.0
)

results, summary_df = compare_dispatch_strategies(
    site_df=site_df,
    solar_forecast_mw=solar_fc_scaled,
    load_forecast_mw=site_df['load_forecast_mw'],
    battery_cfg=battery_cfg
)
display(summary_df)


[2026-10-25 13:11:13] [INFO] [src.preprocessing]: Loading cached clean data from /home/asus/Projects/Renewable-Energy-Intelligence/data/processed/DK_1_clean.parquet


,Strategy,Total Cost (EUR),Savings vs No Battery (EUR),Savings vs No Battery (%),Savings vs Rule-Based (EUR),Energy Charged (MWh),Energy Discharged (MWh),Grid Import (MWh),Grid Export (MWh),Curtailment (MWh),Eq. Full Cycles,Max Balance Residual (MW),Violations
0,No Battery Baseline,754459.45,0.00,0.00,-20430.66,0.00,0.00,16835.54,120.30,8.42,0.00,0.0,0
1,Rule-Based Strategy,734028.79,20430.66,2.71,0.00,1917.95,1687.15,16952.53,14.90,0.00,180.26,0.0,0
2,Forecast-Informed (LP),712738.52,41720.93,5.53,21290.27,3306.34,2909.58,17259.09,153.06,2.45,310.80,0.0,0
3,Perfect Foresight (Reference),710237.44,44222.01,5.86,23791.35,3294.79,2899.42,17108.19,5.04,0.94,309.71,0.0,0


### 1. Invariant & Constraint Verification
- Energy balance residuals must be ~0.
- Constraint violations must be 0.


In [2]:
for name, r in results.items():
    print(f"Strategy: {r.strategy_name:<32} | Violations: {r.constraint_violations} | Max Residual: {r.max_energy_balance_residual:.6f} MW")
    assert r.constraint_violations == 0
    assert r.max_energy_balance_residual < 1e-4
print("All physical and operational constraints passed with zero violations!")


Strategy: No Battery Baseline              | Violations: 0 | Max Residual: 0.000000 MW
Strategy: Rule-Based Baseline              | Violations: 0 | Max Residual: 0.000000 MW
Strategy: Forecast-Informed Dispatch       | Violations: 0 | Max Residual: 0.000000 MW
Strategy: Perfect-Foresight (Upper Bound)  | Violations: 0 | Max Residual: 0.000000 MW
All physical and operational constraints passed with zero violations!
